In [4]:

import pandas as pd

# Load the dataset
df = pd.read_csv('/content/student_exam_performance.csv')

# Check the dataset size
print("Dataset Shape:", df.shape)

# Display the first five rows
display(df.head())


Dataset Shape: (100000, 44)


,student_id,age,gender,education_level,school_type,family_income,parent_education,urban_rural,previous_exam_score,previous_gpa,...,exam_difficulty,exam_preparation_days,questions_attempted,questions_correct,time_management_score,exam_anxiety_level,exam_score,performance_grade,pass_status,performance_level
0,STU_000001,20,Female,High School,Public,Middle,NaN,Rural,78.36,2.95,...,Medium,27,95,86,NaN,9.39,90.40,A,Pass,High
1,STU_000002,17,Male,High School,Public,Middle,NaN,Suburban,73.40,2.89,...,Easy,4,100,86,87.17,6.31,86.21,A,Pass,High
2,STU_000003,18,Other,Undergraduate,Public,Middle,Master,Urban,82.76,3.35,...,Easy,28,96,74,63.77,9.87,76.11,B,Pass,Medium
3,STU_000004,20,Male,High School,Public,Middle,High School,Suburban,60.61,2.57,...,Medium,16,97,77,59.51,9.68,79.74,B,Pass,Medium
4,STU_000005,16,Female,High School,Private,High,Bachelor,Suburban,74.79,2.84,...,Medium,28,98,52,84.64,10.00,50.66,D,Pass,Low


In [5]:

# Step 4: Data Quality Assessment

print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])

print("\nTotal Missing Values:")
print(df.isnull().sum().sum())

print("\nMissing Values by Column:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate Rows:")
print(df.duplicated().sum())

print("\nData Types:")
print(df.dtypes)


Total Rows: 100000
Total Columns: 44

Total Missing Values:
52161

Missing Values by Column:
attendance_percentage         9863
time_management_score         9613
notes_quality                 8407
previous_gpa                  7793
sleep_quality                 6994
parent_education              6526
device_availability           2965
school_type                      0
education_level                  0
gender                           0
age                              0
student_id                       0
assignment_completion_rate       0
previous_exam_score              0
family_income                    0
urban_rural                      0
online_learning_hours            0
study_consistency                0
class_participation              0
study_hours_per_day              0
study_method                     0
study_environment                0
practice_tests_completed         0
revision_frequency               0
sleep_hours                      0
daily_screen_time               

In [6]:

# Step 5: Handle Missing Values

df_clean = df.copy()

numeric_cols = [
    'attendance_percentage',
    'time_management_score',
    'previous_gpa'
]

categorical_cols = [
    'notes_quality',
    'sleep_quality',
    'parent_education',
    'device_availability'
]

# Fill numeric missing values with median
for col in numeric_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].median()
    )

# Fill categorical missing values with mode
for col in categorical_cols:
    df_clean[col] = df_clean[col].fillna(
        df_clean[col].mode()[0]
    )

print("Rows after cleaning:", df_clean.shape[0])
print("Columns after cleaning:", df_clean.shape[1])
print("Missing values after cleaning:",
      df_clean.isnull().sum().sum())


Rows after cleaning: 100000
Columns after cleaning: 44
Missing values after cleaning: 0


In [7]:

# Step 6: Check numeric ranges

columns_to_check = [
    'age',
    'attendance_percentage',
    'study_hours_per_day',
    'previous_exam_score',
    'previous_gpa',
    'assignment_completion_rate',
    'sleep_hours',
    'time_management_score'
]

print("Numeric Data Summary:")
display(
    df_clean[columns_to_check].describe().T[
        ['min', 'max', 'mean', '50%']
    ]
)


Numeric Data Summary:


,min,max,mean,50%
age,14.00,20.0,17.005740,17.00
attendance_percentage,46.34,100.0,84.630011,84.75
study_hours_per_day,0.50,12.0,2.995725,2.67
previous_exam_score,30.00,100.0,69.531033,69.53
previous_gpa,1.00,4.0,2.780823,2.78
assignment_completion_rate,20.00,100.0,69.724405,69.57
sleep_hours,3.00,11.0,6.995102,7.00
time_management_score,17.03,100.0,78.456368,79.00


In [8]:

# Step 7: Feature Selection

selected_features = [
    'attendance_percentage',
    'study_hours_per_day',
    'previous_exam_score',
    'previous_gpa',
    'assignment_completion_rate',
    'study_consistency',
    'revision_frequency',
    'motivation_level',
    'time_management_score',
    'sleep_hours'
]

target = 'pass_status'

# Create a dataset with selected features and target
df_selected = df_clean[selected_features + [target]].copy()

print("Rows:", df_selected.shape[0])
print("Columns:", df_selected.shape[1])

print("\nTarget Distribution:")
print(df_selected[target].value_counts())

display(df_selected.head())


Rows: 100000
Columns: 11

Target Distribution:
pass_status
Pass    77389
Fail    22611
Name: count, dtype: int64


,attendance_percentage,study_hours_per_day,previous_exam_score,previous_gpa,assignment_completion_rate,study_consistency,revision_frequency,motivation_level,time_management_score,sleep_hours,pass_status
0,81.49,4.94,78.36,2.95,65.18,Medium,Rarely,High,79.00,6.31,Pass
1,84.75,3.42,73.40,2.89,76.53,Medium,Daily,High,87.17,6.81,Pass
2,100.00,1.06,82.76,3.35,70.39,Low,Weekly,Medium,63.77,6.53,Pass
3,84.91,7.89,60.61,2.57,75.06,Medium,Weekly,High,59.51,6.60,Pass
4,88.21,2.63,74.79,2.84,61.54,Medium,Rarely,High,84.64,7.83,Pass


In [9]:

# Step 8: Prepare Data for Machine Learning

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Use original data to avoid leakage
X = df[selected_features].copy()

# Convert target to binary: Pass=1, Fail=0
y = df['pass_status'].map({
    'Pass': 1,
    'Fail': 0
})

# Check target labels
assert y.notna().all(), "Check pass_status values"

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(
    include=['number']
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=['number']
).columns.tolist()

# Numeric preprocessing
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Categorical preprocessing
categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessing
preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Split data before fitting preprocessing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Rows:", X_train.shape[0])
print("Testing Rows:", X_test.shape[0])
print("Numeric Features:", numeric_features)
print("Categorical Features:", categorical_features)
print("Target Distribution:")
print(y.value_counts())


Training Rows: 80000
Testing Rows: 20000
Numeric Features: ['attendance_percentage', 'study_hours_per_day', 'previous_exam_score', 'previous_gpa', 'assignment_completion_rate', 'time_management_score', 'sleep_hours']
Categorical Features: ['study_consistency', 'revision_frequency', 'motivation_level']
Target Distribution:
pass_status
1    77389
0    22611
Name: count, dtype: int64


In [10]:

# Step 9: Logistic Regression Model

from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Create model with preprocessing
logistic_model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Train model
logistic_model.fit(X_train, y_train)

# Predict test data
y_pred = logistic_model.predict(X_test)

# Evaluate model
print("Logistic Regression Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=['Fail', 'Pass'],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Logistic Regression Accuracy:
0.81095

Classification Report:
              precision    recall  f1-score   support

        Fail       0.65      0.36      0.46      4522
        Pass       0.83      0.94      0.89     15478

    accuracy                           0.81     20000
   macro avg       0.74      0.65      0.67     20000
weighted avg       0.79      0.81      0.79     20000


Confusion Matrix:
[[ 1606  2916]
 [  865 14613]]


In [11]:

# Step 10: Logistic Regression Cross Validation

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer, recall_score

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    'accuracy': 'accuracy',
    'f1_fail': make_scorer(
        recall_score,
        pos_label=0
    )
}

scores = cross_validate(
    logistic_model,
    X_train,
    y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=-1
)

print("Accuracy per fold:",
      scores['test_accuracy'])

print("Mean Accuracy:",
      scores['test_accuracy'].mean())

print("Fail Recall per fold:",
      scores['test_f1_fail'])

print("Mean Fail Recall:",
      scores['test_f1_fail'].mean())


Accuracy per fold: [0.807875  0.8116875 0.8103125 0.80725   0.808875 ]
Mean Accuracy: 0.8091999999999999
Fail Recall per fold: [0.34752557 0.36069652 0.34687673 0.33996683 0.35903814]
Mean Fail Recall: 0.35082075873818014


In [12]:

# Step 11: Decision Tree Model

from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

decision_tree_model = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', DecisionTreeClassifier(
        max_depth=6,
        min_samples_leaf=20,
        random_state=42
    ))
])

# Train model
decision_tree_model.fit(X_train, y_train)

# Predict test data
tree_pred = decision_tree_model.predict(X_test)

# Evaluate model
print("Decision Tree Accuracy:")
print(accuracy_score(y_test, tree_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    tree_pred,
    target_names=['Fail', 'Pass'],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, tree_pred))


Decision Tree Accuracy:
0.80075

Classification Report:
              precision    recall  f1-score   support

        Fail       0.61      0.32      0.42      4522
        Pass       0.83      0.94      0.88     15478

    accuracy                           0.80     20000
   macro avg       0.72      0.63      0.65     20000
weighted avg       0.78      0.80      0.78     20000


Confusion Matrix:
[[ 1462  3060]
 [  925 14553]]


In [13]:

# Step 13: Logistic Regression Hyperparameter Tuning

from sklearn.model_selection import GridSearchCV
from sklearn.metrics import make_scorer, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Create model
tuning_pipeline = Pipeline([
    ('preprocessing', preprocessor),
    ('classifier', LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

# Parameters to test
param_grid = {
    'classifier__C': [0.1, 1, 10],
    'classifier__class_weight': [None, 'balanced']
}

# Focus on detecting Fail students (label 0)
fail_recall = make_scorer(
    recall_score,
    pos_label=0
)

# Grid Search with 5-fold cross validation
grid_search = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    scoring=fail_recall,
    cv=5,
    n_jobs=-1
)

# Train on training data only
grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross Validation Fail Recall:")
print(grid_search.best_score_)


Best Parameters:
{'classifier__C': 1, 'classifier__class_weight': 'balanced'}

Best Cross Validation Fail Recall:
0.7452044144466742


In [14]:

# Step 14: Evaluate Tuned Logistic Regression

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# Best model from GridSearchCV
best_logistic_model = grid_search.best_estimator_

# Predict test data
tuned_pred = best_logistic_model.predict(X_test)

# Evaluate
print("Tuned Logistic Regression Accuracy:")
print(accuracy_score(y_test, tuned_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    tuned_pred,
    target_names=['Fail', 'Pass'],
    zero_division=0
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, tuned_pred))


Tuned Logistic Regression Accuracy:
0.72665

Classification Report:
              precision    recall  f1-score   support

        Fail       0.44      0.75      0.55      4522
        Pass       0.91      0.72      0.80     15478

    accuracy                           0.73     20000
   macro avg       0.67      0.73      0.68     20000
weighted avg       0.80      0.73      0.75     20000


Confusion Matrix:
[[ 3369  1153]
 [ 4314 11164]]


In [15]:

# Step 15: Final Model Comparison

import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

models_predictions = {
    'Logistic Regression': y_pred,
    'Decision Tree': tree_pred,
    'Tuned Logistic Regression': tuned_pred
}

results = []

for name, predictions in models_predictions.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Fail Precision': precision_score(
            y_test, predictions, pos_label=0
        ),
        'Fail Recall': recall_score(
            y_test, predictions, pos_label=0
        ),
        'Fail F1': f1_score(
            y_test, predictions, pos_label=0
        )
    })

comparison_df = pd.DataFrame(results)

display(comparison_df.round(4))


,Model,Accuracy,Fail Precision,Fail Recall,Fail F1
0,Logistic Regression,0.8109,0.6499,0.3552,0.4593
1,Decision Tree,0.8008,0.6125,0.3233,0.4232
2,Tuned Logistic Regression,0.7266,0.4385,0.7450,0.5521


In [16]:

# Step 16: Save Final Model

import joblib

# Save trained model with preprocessing
joblib.dump(
    best_logistic_model,
    'student_outcome_model.pkl'
)

print("Model saved successfully!")


Model saved successfully!


In [17]:

# Step 17: Install Streamlit

!pip install streamlit


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 22.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 43.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [streamlit]


In [18]:

# Step 18: Create Streamlit App

%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(
    page_title="Student Academic Outcomes",
    page_icon="🎓"
)

st.title("🎓 Student Academic Outcomes Prediction")
st.write("Enter student information to predict Pass or Fail.")

# Load trained model
model = joblib.load("student_outcome_model.pkl")

attendance = st.slider("Attendance Percentage", 0, 100, 85)
study_hours = st.slider("Study Hours Per Day", 0.0, 12.0, 3.0)
previous_score = st.slider("Previous Exam Score", 0, 100, 70)
previous_gpa = st.slider("Previous GPA", 1.0, 4.0, 2.8)
assignment = st.slider("Assignment Completion Rate", 0, 100, 70)
time_management = st.slider("Time Management Score", 0, 100, 78)
sleep_hours = st.slider("Sleep Hours", 0.0, 12.0, 7.0)

# Use original training categories
study_consistency = st.selectbox(
    "Study Consistency",
    sorted(["Low", "Medium", "High"])
)

revision_frequency = st.selectbox(
    "Revision Frequency",
    sorted(["Low", "Medium", "High"])
)

motivation_level = st.selectbox(
    "Motivation Level",
    sorted(["Low", "Medium", "High"])
)

input_data = pd.DataFrame([{
    "attendance_percentage": attendance,
    "study_hours_per_day": study_hours,
    "previous_exam_score": previous_score,
    "previous_gpa": previous_gpa,
    "assignment_completion_rate": assignment,
    "study_consistency": study_consistency,
    "revision_frequency": revision_frequency,
    "motivation_level": motivation_level,
    "time_management_score": time_management,
    "sleep_hours": sleep_hours
}])

if st.button("Predict Student Outcome"):
    prediction = model.predict(input_data)[0]

    if prediction == 1:
        st.success("Predicted Outcome: PASS 🎉")
    else:
        st.error("Predicted Outcome: FAIL ⚠️")


Writing app.py


In [19]:

# Step 19: Check Categories and Run Streamlit

# Check actual category values
for col in [
    'study_consistency',
    'revision_frequency',
    'motivation_level'
]:
    print(col, ":", df[col].dropna().unique().tolist())

# Start Streamlit
import subprocess

streamlit_process = subprocess.Popen([
    "streamlit",
    "run",
    "app.py",
    "--server.port",
    "8501",
    "--server.headless",
    "true"
])

print("\nStreamlit started on port 8501")


study_consistency : ['Medium', 'Low', 'High']
revision_frequency : ['Rarely', 'Daily', 'Weekly']
motivation_level : ['High', 'Medium', 'Low']

Streamlit started on port 8501


In [20]:

# Step 20: Fix Revision Frequency Options

from pathlib import Path

app_path = Path("app.py")
app_code = app_path.read_text()

old_code = '''revision_frequency = st.selectbox(
    "Revision Frequency",
    sorted(["Low", "Medium", "High"])
)'''

new_code = '''revision_frequency = st.selectbox(
    "Revision Frequency",
    ["Rarely", "Daily", "Weekly"]
)'''

if old_code in app_code:
    app_code = app_code.replace(old_code, new_code)
    app_path.write_text(app_code)
    print("App updated successfully!")
else:
    print("Check app.py: original code not found.")


App updated successfully!


In [21]:

# Step 21: Open Streamlit App in Colab

from google.colab import output

output.serve_kernel_port_as_window(8501)


Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [22]:

# Open Streamlit inside Google Colab

from google.colab import output

output.serve_kernel_port_as_iframe(8501)



<IPython.core.display.Javascript object>

In [23]:

# Step 22: Check Streamlit Status

import subprocess

result = subprocess.run(
    ["curl", "-sS", "-I", "--max-time", "5",
     "http://localhost:8501"],
    capture_output=True,
    text=True
)

print("Streamlit Response:")
print(result.stdout)
print(result.stderr)

print("Streamlit Process Status:")
print(streamlit_process.poll())


Streamlit Response:
HTTP/1.1 200 OK
date: Tue, 17 Nov 2026 08:07:06 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 7863
last-modified: Tue, 17 Nov 2026 08:02:34 GMT
etag: "6809e4a2046340089032a25a68e2f919"
cache-control: no-cache



Streamlit Process Status:
None


In [24]:

# Step 23: Check Streamlit Errors

import subprocess

result = subprocess.run(
    ["ps", "aux"],
    capture_output=True,
    text=True
)

for line in result.stdout.splitlines():
    if "streamlit" in line.lower():
        print(line)

print("\nModel File Check:")

import os

print("app.py exists:", os.path.exists("app.py"))
print(
    "student_outcome_model.pkl exists:",
    os.path.exists("student_outcome_model.pkl")
)

print("\nStreamlit Process:")
print("Return code:", streamlit_process.poll())


root        6911  2.4  0.5 1006048 78224 ?       Sl   08:03   0:05 /usr/bin/python3 /usr/local/bin/streamlit run app.py --server.port 8501 --server.headless true

Model File Check:
app.py exists: True
student_outcome_model.pkl exists: True

Streamlit Process:
Return code: None


In [25]:

# Step 24: Test Model Loading

import joblib
import pandas as pd

try:
    model = joblib.load("student_outcome_model.pkl")
    print("Model loaded successfully!")

    sample = X_test.head(1)
    prediction = model.predict(sample)

    print("Prediction:", prediction[0])
    print("Model prediction works!")

except Exception as e:
    print("ERROR:", type(e).__name__)
    print(e)


Model loaded successfully!
Prediction: 1
Model prediction works!


In [26]:

# Step 25: Open Streamlit using Colab Proxy

from google.colab import output

url = output.eval_js(
    "google.colab.kernel.proxyPort(8501)"
)

print("Open your Streamlit app here:")
print(url)


Open your Streamlit app here:
https://8501-m-s-usc1a1-h9b8kl6tk2pe-a.us-central1.prod.r.colab.dev


In [27]:

# Step 26: Check Streamlit Health

import requests

try:
    response = requests.get(
        "http://localhost:8501/_stcore/health",
        timeout=10
    )

    print("Status Code:", response.status_code)
    print("Response:", response.text)

except Exception as e:
    print("Error:", str(e))



Status Code: 200
Response: ok


In [28]:

# Step 27: Restart Streamlit and Save Logs

import subprocess
import time

# Stop previous Streamlit process
streamlit_process.terminate()

try:
    streamlit_process.wait(timeout=5)
except subprocess.TimeoutExpired:
    streamlit_process.kill()
    streamlit_process.wait()

# Restart and save logs
log_file = open("streamlit.log", "w")

streamlit_process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--logger.level", "debug"
    ],
    stdout=log_file,
    stderr=subprocess.STDOUT
)

time.sleep(5)

print("Process status:", streamlit_process.poll())

print("\nStreamlit Logs:")
with open("streamlit.log") as f:
    print(f.read()[-4000:])


Process status: None

Streamlit Logs:


2026-11-17 08:10:10.842 No ASGI app assignments found in /content/app.py



In [29]:

# Step 28: Inspect Streamlit App

from pathlib import Path
import streamlit as st

print("Streamlit Version:", st.__version__)

print("\nFirst 20 lines of app.py:")
app_lines = Path("app.py").read_text().splitlines()

for number, line in enumerate(app_lines[:20], start=1):
    print(f"{number}: {line}")

print("\nFile size:", Path("app.py").stat().st_size, "bytes")



Streamlit Version: 1.67.0

First 20 lines of app.py:
1: import streamlit as st
2: import pandas as pd
3: import joblib
4: 
5: st.set_page_config(
6:     page_title="Student Academic Outcomes",
7:     page_icon="🎓"
8: )
9: 
10: st.title("🎓 Student Academic Outcomes Prediction")
11: st.write("Enter student information to predict Pass or Fail.")
12: 
13: # Load trained model
14: model = joblib.load("student_outcome_model.pkl")
15: 
16: attendance = st.slider("Attendance Percentage", 0, 100, 85)
17: study_hours = st.slider("Study Hours Per Day", 0.0, 12.0, 3.0)
18: previous_score = st.slider("Previous Exam Score", 0, 100, 70)
19: previous_gpa = st.slider("Previous GPA", 1.0, 4.0, 2.8)
20: assignment = st.slider("Assignment Completion Rate", 0, 100, 70)

File size: 1764 bytes


In [30]:

# Step 29: Check app.py for Python errors

import py_compile

try:
    py_compile.compile("app.py", doraise=True)
    print("SUCCESS: app.py has no Python syntax errors!")
except Exception as e:
    print("ERROR:", e)


SUCCESS: app.py has no Python syntax errors!


In [31]:

# Step 29: Check Python Syntax

import py_compile

try:
    py_compile.compile("app.py", doraise=True)
    print("SUCCESS: app.py has no Python syntax errors!")
except Exception as e:
    print("ERROR:", e)


SUCCESS: app.py has no Python syntax errors!


In [32]:

# Step 30: Create requirements.txt

from pathlib import Path
import sklearn
import pandas
import joblib
import streamlit

requirements = f"""streamlit=={streamlit.__version__}
pandas=={pandas.__version__}
scikit-learn=={sklearn.__version__}
joblib=={joblib.__version__}
"""

Path("requirements.txt").write_text(requirements)

print("requirements.txt created successfully!")
print(requirements)



requirements.txt created successfully!
streamlit==1.67.0
pandas==2.2.3
scikit-learn==1.6.1
joblib==1.6.0



In [33]:

# Step 31: Download Streamlit Project Files

from google.colab import files
import zipfile
import os

project_files = [
    "app.py",
    "student_outcome_model.pkl",
    "requirements.txt"
]

with zipfile.ZipFile("student_project.zip", "w") as zipf:
    for filename in project_files:
        if os.path.exists(filename):
            zipf.write(filename)
            print("Added:", filename)
        else:
            print("Missing:", filename)

files.download("student_project.zip")


Added: app.py
Added: student_outcome_model.pkl
Added: requirements.txt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>